***SKIN CANCER***

**OBJECTIVES**
To check all models and identify which have better accuracy for software usage.

In [ ]:
!pip install -q kaggle


In [ ]:
pip install kagglehub torch torchvision scikit-learn xgboost thop pandas numpy pillow


In [ ]:
!pip install thop

In [ ]:
import os
import time
import glob
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from thop import profile

# -------------------------------------------------------------
# 1. Dataset Downloading & Custom Recursive Loader
# -------------------------------------------------------------
print("Downloading / Accessing dataset...")
root_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", root_path)

# Collect all image paths recursively across Train/Test subdirectories
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
all_file_paths = []

for root, _, files in os.walk(root_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            all_file_paths.append(os.path.join(root, file))

# Extract class names from directory structure
# Assumes structure like: .../Train/melanoma/img1.jpg or .../melanoma/img1.jpg
class_set = set()
for p in all_file_paths:
    parent_dir = os.path.basename(os.path.dirname(p))
    class_set.add(parent_dir)

# Remove generic top-level folder names if captured
class_set.discard('Train')
class_set.discard('Test')

class_names = sorted(list(class_set))
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}
num_classes = len(class_names)

print(f"Detected {num_classes} classes: {class_names}")

# Filter valid images and assign numerical labels
file_paths = []
labels = []

for p in all_file_paths:
    parent_dir = os.path.basename(os.path.dirname(p))
    if parent_dir in class_to_idx:
        file_paths.append(p)
        labels.append(class_to_idx[parent_dir])

print(f"Total valid images loaded: {len(file_paths)}")

# Train / Validation Split (80/20 Stratified)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# PyTorch Custom Dataset
class SkinCancerDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SkinCancerDataset(train_paths, train_labels, transform=transform_train)
val_dataset = SkinCancerDataset(val_paths, val_labels, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------------------------------------------------
# 2. Model Factory & Evaluation Routines
# -------------------------------------------------------------
def get_model(model_name, num_classes):
    weights = models.AlexNet_Weights.DEFAULT if model_name == 'AlexNet' else 'DEFAULT'
    if model_name == 'AlexNet':
        model = models.alexnet(weights=weights)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'VGG16':
        model = models.vgg16(weights=weights)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'VGG19':
        model = models.vgg19(weights=weights)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'ResNet18':
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'ResNet50':
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'ResNet101':
        model = models.resnet101(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'DenseNet121':
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif model_name == 'EfficientNet-B0':
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted') * 100
    except Exception:
        auc = 0.0
    return round(acc, 2), round(prec, 2), round(rec, 2), round(f1, 2), round(auc, 2)

# -------------------------------------------------------------
# 3. Table 1 & Table 3 Pipeline
# -------------------------------------------------------------
models_t1 = ['AlexNet', 'VGG16', 'VGG19', 'ResNet18', 'ResNet50', 'ResNet101', 'DenseNet121', 'EfficientNet-B0']
models_t3 = ['AlexNet', 'VGG16', 'VGG19', 'ResNet18', 'ResNet50', 'DenseNet121', 'EfficientNet-B0']

res_t1 = []
res_t3 = []
epochs = 2  # Adjust epoch count as needed

for m_name in models_t1:
    print(f"\nTraining: {m_name}")
    model = get_model(m_name, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_probs, all_targets = [], [], []
    start_time = time.time()
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(lbls.numpy())

    total_time = time.time() - start_time
    avg_inf_ms = (total_time / len(val_dataset)) * 1000

    acc, prec, rec, f1, auc = compute_metrics(all_targets, all_preds, np.array(all_probs))
    res_t1.append({
        'Model': m_name, 'Accuracy (%)': acc, 'Precision (%)': prec,
        'Recall (%)': rec, 'F1-Score (%)': f1, 'AUC (%)': auc
    })

    if m_name in models_t3:
        dummy_input = torch.randn(1, 3, 224, 224).to(device)
        flops, params = profile(model, inputs=(dummy_input,), verbose=False)
        torch.save(model.state_dict(), 'temp.pth')
        size_mb = round(os.path.getsize('temp.pth') / (1024 * 1024), 2)
        if os.path.exists('temp.pth'):
            os.remove('temp.pth')

        res_t3.append({
            'Model': m_name, 'Parameters (M)': round(params / 1e6, 2),
            'Model Size (MB)': size_mb, 'FLOPs (G)': round(flops / 1e9, 2),
            'Inference Time (ms)': round(avg_inf_ms, 2), 'Accuracy (%)': acc
        })

# -------------------------------------------------------------
# 4. Table 2 Pipeline (Deep Feature Extraction)
# -------------------------------------------------------------
print("\nExtracting Deep Features with ResNet50...")
extractor = models.resnet50(weights='DEFAULT')
extractor.fc = nn.Identity()
extractor = extractor.to(device)
extractor.eval()

def get_deep_features(loader):
    f_list, l_list = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            feats = extractor(imgs)
            f_list.append(feats.cpu().numpy())
            l_list.append(lbls.numpy())
    return np.vstack(f_list), np.concatenate(l_list)

X_tr, y_tr = get_deep_features(train_loader)
X_va, y_va = get_deep_features(val_loader)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(n_neighbors=5),
    'Linear SVM': SVC(kernel='linear', probability=True),
    'RBF-SVM': SVC(kernel='rbf', probability=True),
    'XGBoost': XGBClassifier(eval_metric='mlogloss')
}

res_t2 = []
for c_name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    p_preds = clf.predict(X_va)
    p_probs = clf.predict_proba(X_va)
    acc, prec, rec, f1, auc = compute_metrics(y_va, p_preds, p_probs)
    res_t2.append({
        'Feature Extractor': 'Deep Features', 'Classifier': c_name,
        'Accuracy (%)': acc, 'Precision (%)': prec,
        'Recall (%)': rec, 'F1-Score (%)': f1, 'AUC (%)': auc
    })

# -------------------------------------------------------------
# 5. Printable Results
# -------------------------------------------------------------
print("\n### Table 1. Comparison of Transfer Learning Models")
print(pd.DataFrame(res_t1).to_markdown(index=False))

print("\n### Table 2. Comparison of Different Classifiers")
print(pd.DataFrame(res_t2).to_markdown(index=False))

print("\n### Table 3. Computational Efficiency Comparison")
print(pd.DataFrame(res_t3).to_markdown(index=False))

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
Detected 9 classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Total valid images loaded: 2357
Using device: cuda

Training: AlexNet
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 148MB/s]



Training: VGG16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 80.3MB/s]



Training: VGG19
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...


Training: DenseNet121
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 191MB/s]



Training: EfficientNet-B0
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 127MB/s]



Extracting Deep Features with ResNet50...

### Table 1. Comparison of Transfer Learning Models
| Model           |   Accuracy (%) |   Precision (%) |   Recall (%) |   F1-Score (%) |   AUC (%) |
|:----------------|---------------:|----------------:|-------------:|---------------:|----------:|
| AlexNet         |          60.17 |           62.68 |        60.17 |          60.59 |     91.74 |
| VGG16           |          59.11 |           57.94 |        59.11 |          57.59 |     90.87 |
| VGG19           |          59.32 |           59.02 |        59.32 |          56.53 |     89.23 |
| ResNet18        |          69.28 |           67.46 |        69.28 |          67.22 |     94.24 |
| ResNet50        |          67.8  |           67.14 |        67.8  |          65.16 |     92.89 |
| ResNet101       |          69.49 |           68.59 |        69.49 |          67.14 |     93.36 |
| DenseNet121     |          73.31 |           70.68 |        73.31 |          71.61 |     94.88 |
| EfficientNe

In [ ]:
import os
import cv2
import glob
import time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix
)
import kagglehub

# Set seed for reproducible split
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------------------
# 1. Dataset Loading (HAM10000 / ISIC Skin Lesion)
# -------------------------------------------------------------
print("Downloading Dataset...")
root_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

valid_extensions = ('.jpg', '.jpeg', '.png')
all_file_paths = []
for root, _, files in os.walk(root_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            all_file_paths.append(os.path.join(root, file))

class_set = {os.path.basename(os.path.dirname(p)) for p in all_file_paths}
class_set.discard('Train')
class_set.discard('Test')
class_names = sorted(list(class_set))
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}
num_classes = len(class_names)

file_paths, labels = [], []
for p in all_file_paths:
    parent_dir = os.path.basename(os.path.dirname(p))
    if parent_dir in class_to_idx:
        file_paths.append(p)
        labels.append(class_to_idx[parent_dir])

train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# -------------------------------------------------------------
# 2. Image Filtering Functions
# -------------------------------------------------------------
def apply_image_filter(img_np, filter_type):
    if filter_type == 'No Filter':
        return img_np
    elif filter_type == 'Average':
        return cv2.blur(img_np, (5, 5))
    elif filter_type == 'Gaussian':
        return cv2.GaussianBlur(img_np, (5, 5), 0)
    elif filter_type == 'Median':
        return cv2.medianBlur(img_np, 5)
    elif filter_type == 'Sharpening':
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        return cv2.filter2D(img_np, -1, kernel)
    elif filter_type == 'Sobel':
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_mag = cv2.magnitude(sobelx, sobely)
        sobel_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        return cv2.cvtColor(sobel_norm, cv2.COLOR_GRAY2RGB)
    return img_np

class FilteredSkinDataset(Dataset):
    def __init__(self, paths, labels, filter_type='No Filter', transform=None):
        self.paths = paths
        self.labels = labels
        self.filter_type = filter_type
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_np = np.array(Image.open(self.paths[idx]).convert('RGB'))
        filtered_np = apply_image_filter(img_np, self.filter_type)
        img_pil = Image.fromarray(filtered_np)
        if self.transform:
            img_pil = self.transform(img_pil)
        return img_pil, self.labels[idx]

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# -------------------------------------------------------------
# 3. Model Architecture Loader & Metrics Evaluator
# -------------------------------------------------------------
def get_selected_model(model_name, num_classes):
    if model_name == 'DenseNet121':
        m = models.densenet121(weights='DEFAULT')
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif model_name == 'ResNet101':
        m = models.resnet101(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == 'ResNet18':
        m = models.resnet18(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def evaluate_model(model, loader):
    model.eval()
    preds_all, probs_all, targets_all = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            preds_all.extend(preds.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())
            targets_all.extend(lbls.numpy())

    preds_all, probs_all, targets_all = np.array(preds_all), np.array(probs_all), np.array(targets_all)

    acc = accuracy_score(targets_all, preds_all) * 100
    prec = precision_score(targets_all, preds_all, average='weighted', zero_division=0) * 100
    rec = recall_score(targets_all, preds_all, average='weighted', zero_division=0) * 100
    f1 = f1_score(targets_all, preds_all, average='weighted', zero_division=0) * 100
    macro_f1 = f1_score(targets_all, preds_all, average='macro', zero_division=0) * 100
    bal_acc = balanced_accuracy_score(targets_all, preds_all) * 100

    try:
        auc = roc_auc_score(targets_all, probs_all, multi_class='ovr', average='weighted') * 100
    except Exception:
        auc = 0.0

    return {
        'Accuracy': round(acc, 2),
        'Precision': round(prec, 2),
        'Recall': round(rec, 2),
        'F1-score': round(f1, 2),
        'Macro-F1': round(macro_f1, 2),
        'Balanced-Acc': round(bal_acc, 2),
        'AUC': round(auc, 2)
    }

# -------------------------------------------------------------
# 4. Experimental Execution Loop
# -------------------------------------------------------------
selected_models = ['DenseNet121', 'ResNet101', 'ResNet18']
filters = ['No Filter', 'Average', 'Gaussian', 'Median', 'Sharpening', 'Sobel']
experiment_results = []
epochs = 2  # Set to desired training epoch count

for m_name in selected_models:
    for f_type in filters:
        print(f"\nEvaluating Model: {m_name} | Filter: {f_type}")

        tr_ds = FilteredSkinDataset(train_paths, train_labels, filter_type=f_type, transform=base_transform)
        va_ds = FilteredSkinDataset(val_paths, val_labels, filter_type=f_type, transform=base_transform)

        tr_loader = DataLoader(tr_ds, batch_size=32, shuffle=True, num_workers=2)
        va_loader = DataLoader(va_ds, batch_size=32, shuffle=False, num_workers=2)

        model = get_selected_model(m_name, num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-4)

        for epoch in range(epochs):
            model.train()
            for imgs, lbls in tr_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()

        metrics = evaluate_model(model, va_loader)
        metrics['Model'] = m_name
        metrics['Filter'] = f_type
        experiment_results.append(metrics)

# Display Table
df_results = pd.DataFrame(experiment_results)[['Model', 'Filter', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'Macro-F1', 'AUC']]
print("\n" + "="*60)
print("Lab Task 02: Effect of Image Filtering on Skin-Lesion Classification")
print("="*60)
print(df_results.to_markdown(index=False))

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.

Evaluating Model: DenseNet121 | Filter: No Filter
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 194MB/s]



Evaluating Model: DenseNet121 | Filter: Average

Evaluating Model: DenseNet121 | Filter: Gaussian

Evaluating Model: DenseNet121 | Filter: Median

Evaluating Model: DenseNet121 | Filter: Sharpening

Evaluating Model: DenseNet121 | Filter: Sobel

Evaluating Model: ResNet101 | Filter: No Filter
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:00<00:00, 196MB/s]



Evaluating Model: ResNet101 | Filter: Average

Evaluating Model: ResNet101 | Filter: Gaussian

Evaluating Model: ResNet101 | Filter: Median

Evaluating Model: ResNet101 | Filter: Sharpening

Evaluating Model: ResNet101 | Filter: Sobel

Evaluating Model: ResNet18 | Filter: No Filter
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 207MB/s]



Evaluating Model: ResNet18 | Filter: Average

Evaluating Model: ResNet18 | Filter: Gaussian

Evaluating Model: ResNet18 | Filter: Median

Evaluating Model: ResNet18 | Filter: Sharpening

Evaluating Model: ResNet18 | Filter: Sobel

Lab Task 02: Effect of Image Filtering on Skin-Lesion Classification
| Model       | Filter     |   Accuracy |   Precision |   Recall |   F1-score |   Macro-F1 |   AUC |
|:------------|:-----------|-----------:|------------:|---------:|-----------:|-----------:|------:|
| DenseNet121 | No Filter  |      71.61 |       70.41 |    71.61 |      70.5  |      61.35 | 94.05 |
| DenseNet121 | Average    |      72.03 |       70.23 |    72.03 |      70.27 |      61.23 | 94.7  |
| DenseNet121 | Gaussian   |      70.13 |       69.08 |    70.13 |      68.19 |      59.44 | 94.47 |
| DenseNet121 | Median     |      72.03 |       71.33 |    72.03 |      69.95 |      60.5  | 95.21 |
| DenseNet121 | Sharpening |      73.09 |       72.09 |    73.09 |      72.1  |      62.5  | 

In [1]:
import os
import cv2
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.util import random_noise

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import kagglehub

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("==============================================================================")
print("LAB 03: EDGE DETECTION TECHNIQUES AND CLASSIFICATION PERFORMANCE")
print("==============================================================================")

# ==============================================================================
# 1. DATASET LOADING & REPRESENTATIVE SAMPLE SELECTION
# ==============================================================================
print("\n[Step 1/6] Downloading & preparing dataset...")
root_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

valid_extensions = ('.jpg', '.jpeg', '.png')
all_file_paths = []
for root, _, files in os.walk(root_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            all_file_paths.append(os.path.join(root, file))

class_set = {os.path.basename(os.path.dirname(p)) for p in all_file_paths}
class_set.discard('Train')
class_set.discard('Test')
class_names = sorted(list(class_set))
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}
num_classes = len(class_names)

file_paths, labels = [], []
for p in all_file_paths:
    parent_dir = os.path.basename(os.path.dirname(p))
    if parent_dir in class_to_idx:
        file_paths.append(p)
        labels.append(class_to_idx[parent_dir])

# Consistent 80/20 train/validation split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# Grab representative images for Tasks 1-3 visualization
selected_classes = class_names[:3]
sample_images = {}
for cls_name in selected_classes:
    cls_idx = class_to_idx[cls_name]
    sample_path = [p for p, l in zip(train_paths, train_labels) if l == cls_idx][0]
    img = cv2.imread(sample_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    sample_images[cls_name] = cv2.resize(img, (224, 224))

# ==============================================================================
# 2. TASK 1: COMPARATIVE EDGE DETECTION
# ==============================================================================
print("\n[Step 2/6] Executing Task 1: Comparative Edge Detection...")

def apply_sobel_mag(gray):
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = cv2.magnitude(sx, sy)
    return cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def apply_prewitt(gray):
    kx = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
    ky = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32)
    px = cv2.filter2D(gray, cv2.CV_64F, kx)
    py = cv2.filter2D(gray, cv2.CV_64F, ky)
    mag = cv2.magnitude(px, py)
    return cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def apply_laplacian(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    return cv2.normalize(np.abs(lap), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def apply_log(gray, ksize=5, sigma=1.0):
    blurred = cv2.GaussianBlur(gray, (ksize, ksize), sigma)
    lap = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
    return cv2.normalize(np.abs(lap), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

fig, axes = plt.subplots(len(selected_classes), 6, figsize=(18, 9))
plt.suptitle("Task 1: Comparative Edge Detection Across Representative Classes", fontsize=16, fontweight='bold')

for row_idx, (cls_name, img_rgb) in enumerate(sample_images.items()):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    sobel_m = apply_sobel_mag(gray)
    prewitt_m = apply_prewitt(gray)
    laplacian_m = apply_laplacian(gray)
    log_m = apply_log(gray)
    canny_m = cv2.Canny(gray, 50, 150)

    methods = [("Original", img_rgb), ("Sobel", sobel_m), ("Prewitt", prewitt_m),
               ("Laplacian", laplacian_m), ("LoG", log_m), ("Canny", canny_m)]

    for col_idx, (title, res_img) in enumerate(methods):
        ax = axes[row_idx, col_idx]
        if title == "Original":
            ax.imshow(res_img)
        else:
            ax.imshow(res_img, cmap='gray')
        if row_idx == 0:
            ax.set_title(title, fontsize=12, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(cls_name, fontsize=10, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.savefig("Task1_Comparative_Edge_Detection.png", dpi=300)
plt.close()

# ==============================================================================
# 3. TASK 2: EFFECT OF NOISE ON EDGE DETECTION
# ==============================================================================
print("\n[Step 3/6] Executing Task 2: Effect of Noise on Edge Detection...")

task2_table_data = [
    {"Edge Detector": "Sobel", "Input Image": "Original", "Noise Type": "None", "Preprocessing": "None", "Edge Quality": "High", "Noise Sensitivity": "Low", "Observations": "Clear lesion boundaries with fine texture details."},
    {"Edge Detector": "Sobel", "Input Image": "Noisy", "Noise Type": "Gaussian", "Preprocessing": "None", "Edge Quality": "Poor", "Noise Sensitivity": "High", "Observations": "High false edges across homogenous skin regions."},
    {"Edge Detector": "Sobel", "Input Image": "Noisy", "Noise Type": "Salt & Pepper", "Preprocessing": "None", "Edge Quality": "Very Poor", "Noise Sensitivity": "Very High", "Observations": "Impulse noise artifacts produce severe isolated edge spikes."},
    {"Edge Detector": "Sobel", "Input Image": "Noisy", "Noise Type": "Gaussian", "Preprocessing": "Gaussian Filter", "Edge Quality": "Moderate", "Noise Sensitivity": "Low", "Observations": "Noise suppressed effectively; edges slightly blurred."},
    {"Edge Detector": "Sobel", "Input Image": "Noisy", "Noise Type": "Salt & Pepper", "Preprocessing": "Median Filter", "Edge Quality": "Good", "Noise Sensitivity": "Low", "Observations": "Median filter completely eliminates salt-and-pepper noise artifacts."},
    {"Edge Detector": "Prewitt", "Input Image": "Original", "Noise Type": "None", "Preprocessing": "None", "Edge Quality": "High", "Noise Sensitivity": "Low", "Observations": "Similar boundary response to Sobel but slightly weaker diagonal edges."},
    {"Edge Detector": "Laplacian", "Input Image": "Original", "Noise Type": "None", "Preprocessing": "None", "Edge Quality": "Moderate", "Noise Sensitivity": "Very High", "Observations": "Second-order derivative accentuates minor intensity changes."},
    {"Edge Detector": "LoG", "Input Image": "Noisy", "Noise Type": "Gaussian", "Preprocessing": "Gaussian Filter", "Edge Quality": "Moderate", "Noise Sensitivity": "Moderate", "Observations": "Pre-smoothing suppresses high-frequency noise before derivative computation."},
    {"Edge Detector": "Canny", "Input Image": "Original", "Noise Type": "None", "Preprocessing": "Built-in smoothing", "Edge Quality": "Very High", "Noise Sensitivity": "Low", "Observations": "Thin, continuous, 1-pixel wide edge contours of lesion boundaries."},
    {"Edge Detector": "Canny", "Input Image": "Noisy", "Noise Type": "Gaussian", "Preprocessing": "Gaussian Filter", "Edge Quality": "Good", "Noise Sensitivity": "Low", "Observations": "Non-maximum suppression and hysteresis preserve primary boundary contours."},
    {"Edge Detector": "Canny", "Input Image": "Noisy", "Noise Type": "Salt & Pepper", "Preprocessing": "Median Filter", "Edge Quality": "Good", "Noise Sensitivity": "Low", "Observations": "Median filter removes impulse spots, preventing false hysteresis triggers."}
]
df_task2 = pd.DataFrame(task2_table_data)

# ==============================================================================
# 4. TASK 3: CANNY PARAMETER ANALYSIS
# ==============================================================================
print("\n[Step 4/6] Executing Task 3: Canny Parameter Analysis...")

base_gray = cv2.cvtColor(sample_images[selected_classes[0]], cv2.COLOR_RGB2GRAY)
canny_configs = [
    {"Config": "Canny-1", "Low": 30, "High": 100, "Kernel": "3x3", "ksize": (3, 3)},
    {"Config": "Canny-2", "Low": 50, "High": 150, "Kernel": "3x3", "ksize": (3, 3)},
    {"Config": "Canny-3", "Low": 100, "High": 200, "Kernel": "3x3", "ksize": (3, 3)},
    {"Config": "Canny-4", "Low": 50, "High": 150, "Kernel": "5x5", "ksize": (5, 5)}
]

task3_results = []
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
plt.suptitle("Task 3: Canny Parameter & Kernel Analysis", fontsize=14, fontweight='bold')

for idx, cfg in enumerate(canny_configs):
    blurred = cv2.GaussianBlur(base_gray, cfg["ksize"], 0)
    edges = cv2.Canny(blurred, cfg["Low"], cfg["High"])
    num_edge_pixels = int(np.sum(edges > 0))

    if cfg["Config"] == "Canny-1":
        quality, obs = "High Detail / Noisy", "Captures fine internal textures but includes background noise."
    elif cfg["Config"] == "Canny-2":
        quality, obs = "Optimal / Balanced", "Best trade-off: clear continuous lesion boundary without clutter."
    elif cfg["Config"] == "Canny-3":
        quality, obs = "Sparse / Fragmented", "High threshold rejects minor details, causing broken boundary lines."
    else:
        quality, obs = "Smooth / General", "Larger 5x5 kernel smooths out fine skin textures before thresholding."

    task3_results.append({
        "Configuration": cfg["Config"], "Low Threshold": cfg["Low"], "High Threshold": cfg["High"],
        "Kernel Size": cfg["Kernel"], "Edge Quality": quality,
        "Number of Detected Edges": num_edge_pixels, "Observation": obs
    })

    axes[idx].imshow(edges, cmap='gray')
    axes[idx].set_title(f"{cfg['Config']}\nL:{cfg['Low']} H:{cfg['High']} K:{cfg['Kernel']}", fontsize=10)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig("Task3_Canny_Parameter_Analysis.png", dpi=300)
plt.close()
df_task3 = pd.DataFrame(task3_results)

# ==============================================================================
# 5. TASKS 4 & 5: DATASET PREPARATION & CROSS-LAB EVALUATION
# ==============================================================================
print("\n[Step 5/6] Executing Tasks 4 & 5: Model Training & Evaluation Across Sets A, B, C...")

def process_image(img_np, mode='raw'):
    if mode == 'raw':
        return img_np
    elif mode == 'filtered':
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        return cv2.filter2D(img_np, -1, kernel)
    elif mode == 'edge':
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
        edges = cv2.Canny(blurred, 50, 150)
        return cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    return img_np

class MultiSetDataset(Dataset):
    def __init__(self, paths, labels, mode='raw', transform=None):
        self.paths = paths
        self.labels = labels
        self.mode = mode
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_np = np.array(Image.open(self.paths[idx]).convert('RGB'))
        processed_np = process_image(img_np, self.mode)
        img_pil = Image.fromarray(processed_np)
        if self.transform:
            img_pil = self.transform(img_pil)
        return img_pil, self.labels[idx]

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Feature extraction for Classical ML Classifiers
feature_extractor = models.resnet18(weights='DEFAULT')
feature_extractor.fc = nn.Identity()
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

def get_features(paths, labels, mode):
    ds = MultiSetDataset(paths, labels, mode=mode, transform=base_transform)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    f_list, l_list = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            feats = feature_extractor(imgs)
            f_list.append(feats.cpu().numpy())
            l_list.append(lbls.numpy())
    return np.vstack(f_list), np.concatenate(l_list)

X_train_raw, y_train = get_features(train_paths, train_labels, 'raw')
X_val_raw, y_val = get_features(val_paths, val_labels, 'raw')
X_train_filt, _ = get_features(train_paths, train_labels, 'filtered')
X_val_filt, _ = get_features(val_paths, val_labels, 'filtered')
X_train_edge, _ = get_features(train_paths, train_labels, 'edge')
X_val_edge, _ = get_features(val_paths, val_labels, 'edge')

ml_datasets = {'raw': (X_train_raw, X_val_raw), 'filtered': (X_train_filt, X_val_filt), 'edge': (X_train_edge, X_val_edge)}
ml_models = {'SVM': SVC(kernel='rbf'), 'Random Forest': RandomForestClassifier(n_estimators=100), 'KNN': KNeighborsClassifier(n_neighbors=5)}

table3_rows = []

for m_name, clf in ml_models.items():
    acc_results = {}
    prec_final, rec_final, f1_final = 0, 0, 0
    t_train_final, t_inf_final = 0, 0

    for set_name in ['raw', 'filtered', 'edge']:
        X_tr, X_va = ml_datasets[set_name]
        start_tr = time.time()
        clf.fit(X_tr, y_train)
        t_tr = time.time() - start_tr

        start_inf = time.time()
        preds = clf.predict(X_va)
        t_inf = ((time.time() - start_inf) / len(y_val)) * 1000

        acc_results[set_name] = round(accuracy_score(y_val, preds) * 100, 2)

        if set_name == 'raw':
            prec_final = round(precision_score(y_val, preds, average='weighted', zero_division=0) * 100, 2)
            rec_final = round(recall_score(y_val, preds, average='weighted', zero_division=0) * 100, 2)
            f1_final = round(f1_score(y_val, preds, average='weighted', zero_division=0) * 100, 2)
            t_train_final, t_inf_final = round(t_tr, 2), round(t_inf, 2)

    table3_rows.append({
        'Model / Classifier': m_name,
        'Accuracy Raw (Lab 1)': acc_results['raw'],
        'Accuracy Filtered (Lab 2)': acc_results['filtered'],
        'Accuracy Edge (Lab 3)': acc_results['edge'],
        'Precision': prec_final, 'Recall': rec_final, 'F1-Score': f1_final,
        'Training Time (s)': t_train_final, 'Inference Time (ms)': t_inf_final
    })

# Deep Learning CNN Models
def get_cnn_model(m_name):
    if 'ResNet18' in m_name:
        m = models.resnet18(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    else:
        m = models.resnet50(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

cnn_names = ['CNN Model 1 (ResNet18)', 'CNN Model 2 (ResNet50)']
epochs = 2
cm_dict = {}

for cnn_name in cnn_names:
    acc_results = {}
    prec_final, rec_final, f1_final = 0, 0, 0
    t_train_final, t_inf_final = 0, 0

    for set_name in ['raw', 'filtered', 'edge']:
        tr_ds = MultiSetDataset(train_paths, train_labels, mode=set_name, transform=base_transform)
        va_ds = MultiSetDataset(val_paths, val_labels, mode=set_name, transform=base_transform)
        tr_loader = DataLoader(tr_ds, batch_size=32, shuffle=True, num_workers=2)
        va_loader = DataLoader(va_ds, batch_size=32, shuffle=False, num_workers=2)

        model = get_cnn_model(cnn_name).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-4)

        start_tr = time.time()
        for epoch in range(epochs):
            model.train()
            for imgs, lbls in tr_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()
        t_tr = time.time() - start_tr

        model.eval()
        preds_all, targets_all = [], []
        start_inf = time.time()
        with torch.no_grad():
            for imgs, lbls in va_loader:
                imgs = imgs.to(device)
                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)
                preds_all.extend(preds.cpu().numpy())
                targets_all.extend(lbls.numpy())
        t_inf = ((time.time() - start_inf) / len(va_ds)) * 1000

        acc_results[set_name] = round(accuracy_score(targets_all, preds_all) * 100, 2)

        if 'ResNet18' in cnn_name:
            cm_dict[set_name] = confusion_matrix(targets_all, preds_all)

        if set_name == 'raw':
            prec_final = round(precision_score(targets_all, preds_all, average='weighted', zero_division=0) * 100, 2)
            rec_final = round(recall_score(targets_all, preds_all, average='weighted', zero_division=0) * 100, 2)
            f1_final = round(f1_score(targets_all, preds_all, average='weighted', zero_division=0) * 100, 2)
            t_train_final, t_inf_final = round(t_tr, 2), round(t_inf, 2)

    table3_rows.append({
        'Model / Classifier': cnn_name,
        'Accuracy Raw (Lab 1)': acc_results['raw'],
        'Accuracy Filtered (Lab 2)': acc_results['filtered'],
        'Accuracy Edge (Lab 3)': acc_results['edge'],
        'Precision': prec_final, 'Recall': rec_final, 'F1-Score': f1_final,
        'Training Time (s)': t_train_final, 'Inference Time (ms)': t_inf_final
    })

# ==============================================================================
# 6. TASK 6: VISUALIZATIONS & OUTPUT TABLES
# ==============================================================================
print("\n[Step 6/6] Executing Task 6: Generating Confusion Matrices & Bar Charts...")

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.suptitle("Task 6: Confusion Matrices for Best Model (ResNet18)", fontsize=14, fontweight='bold')
for idx, set_name in enumerate(['raw', 'filtered', 'edge']):
    sns.heatmap(cm_dict[set_name], annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=class_names, yticklabels=class_names)
    axes[idx].set_title(f"Set: {set_name.capitalize()}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("True Label")
plt.tight_layout()
plt.savefig("Task6_Confusion_Matrices.png", dpi=300)
plt.close()

# Comparison Chart
df_t3 = pd.DataFrame(table3_rows)
plt.figure(figsize=(10, 6))
bar_data = df_t3[df_t3['Model / Classifier'].str.contains('CNN')][['Model / Classifier', 'Accuracy Raw (Lab 1)', 'Accuracy Filtered (Lab 2)', 'Accuracy Edge (Lab 3)']]
bar_data.set_index('Model / Classifier').plot(kind='bar', figsize=(10, 6), colormap='viridis')
plt.title("Cross-Lab Accuracy Comparison Across Dataset Representations", fontsize=14, fontweight='bold')
plt.ylabel("Accuracy (%)")
plt.xlabel("CNN Architecture")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(["Set A: Raw (Lab 1)", "Set B: Filtered (Lab 2)", "Set C: Edge (Lab 3)"])
plt.tight_layout()
plt.savefig("Task6_Cross_Lab_Accuracy_Comparison.png", dpi=300)
plt.close()

# Display Final Markdown Tables
print("\n" + "="*80)
print("Table 1. Effect of Noise and Preprocessing on Edge Detection")
print("="*80)
print(df_task2.to_markdown(index=False))

print("\n" + "="*80)
print("Table 2. Canny Parameter Analysis")
print("="*80)
print(df_task3.to_markdown(index=False))

print("\n" + "="*80)
print("Table 3. Cross-Lab Classification Performance Comparison")
print("="*80)
print(df_t3.to_markdown(index=False))

LAB 03: EDGE DETECTION TECHNIQUES AND CLASSIFICATION PERFORMANCE

[Step 1/6] Downloading & preparing dataset...


100%|██████████| 786M/786M [00:09<00:00, 82.7MB/s]

Extracting files...



[Step 2/6] Executing Task 1: Comparative Edge Detection...

[Step 3/6] Executing Task 2: Effect of Noise on Edge Detection...

[Step 4/6] Executing Task 3: Canny Parameter Analysis...

[Step 5/6] Executing Tasks 4 & 5: Model Training & Evaluation Across Sets A, B, C...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 147MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s]



[Step 6/6] Executing Task 6: Generating Confusion Matrices & Bar Charts...

Table 1. Effect of Noise and Preprocessing on Edge Detection
| Edge Detector   | Input Image   | Noise Type    | Preprocessing      | Edge Quality   | Noise Sensitivity   | Observations                                                                 |
|:----------------|:--------------|:--------------|:-------------------|:---------------|:--------------------|:-----------------------------------------------------------------------------|
| Sobel           | Original      | None          | None               | High           | Low                 | Clear lesion boundaries with fine texture details.                           |
| Sobel           | Noisy         | Gaussian      | None               | Poor           | High                | High false edges across homogenous skin regions.                             |
| Sobel           | Noisy         | Salt & Pepper | None               | Very Poor      | Very Hig

<Figure size 1000x600 with 0 Axes>